# Rolling beta sensitivity and residual-stationarity p-values

This notebook is a streamlined version of the original analysis. It focuses on the three figures needed for the presentation:

1. rolling coefficient fluctuation,
2. rolling cointegration / residual-stationarity p-value fluctuation,
3. whether current OLS standard error predicts subsequent rolling-beta movement.

The predictive analysis uses **one fixed window length** so the result is not mechanically driven by comparing short windows with long windows.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

import yfinance as yf
from scipy.stats import t as student_t
from statsmodels.tsa.stattools import coint

START_DATE = "2015-01-01"
END_DATE = None

X_TICKER = "HO=F"
Y_TICKER = "RB=F"
USE_LOG_PRICES = True

WINDOW = 120
PRED_HORIZON = 5
LARGE_MOVE_Q = 0.80
SE_BINS = 10
EPS = 1e-12

OUT = Path("presentation_plots")
OUT.mkdir(exist_ok=True)

plt.rcParams.update({
    "figure.figsize": (11, 4.8),
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.18,
    "font.size": 11,
})


In [ ]:
# Download and prepare the pair
raw = yf.download(
    [X_TICKER, Y_TICKER],
    start=START_DATE,
    end=END_DATE,
    auto_adjust=True,
    progress=False,
)

if isinstance(raw.columns, pd.MultiIndex):
    prices = raw["Close"][[X_TICKER, Y_TICKER]].copy()
else:
    prices = raw[["Close"]].copy()
    prices.columns = [X_TICKER, Y_TICKER]

prices = prices.rename(columns={X_TICKER: "x", Y_TICKER: "y"}).dropna()

if USE_LOG_PRICES:
    data = np.log(prices)
else:
    data = prices.copy()

print(data.head())
print(f"Observations: {len(data):,}")


In [ ]:
def ols_window(xw, yw):
    """OLS y = alpha + beta x + u, with the usual homoskedastic beta SE."""
    xw = np.asarray(xw, dtype=float)
    yw = np.asarray(yw, dtype=float)
    n = len(xw)

    xbar = xw.mean()
    ybar = yw.mean()
    xc = xw - xbar
    yc = yw - ybar

    sxx = np.sum(xc**2)
    if n <= 2 or sxx <= EPS:
        return np.nan, np.nan, np.nan, np.nan, np.nan

    beta = np.sum(xc * yc) / sxx
    alpha = ybar - beta * xbar
    resid = yw - alpha - beta * xw

    sigma2 = np.sum(resid**2) / (n - 2)
    beta_se = np.sqrt(sigma2 / sxx)

    if beta_se <= EPS:
        t_stat = np.nan
        beta_p = np.nan
    else:
        t_stat = beta / beta_se
        beta_p = 2 * (1 - student_t.cdf(abs(t_stat), df=n - 2))

    return alpha, beta, beta_se, t_stat, beta_p


def safe_coint_p(yw, xw):
    """Engle-Granger residual-based cointegration p-value (MacKinnon approximation)."""
    try:
        stat, pval, _ = coint(yw, xw, trend="c", autolag="aic")
        return stat, pval
    except Exception:
        return np.nan, np.nan


rows = []
x = data["x"].to_numpy()
y = data["y"].to_numpy()
dates = data.index

for end in range(WINDOW - 1, len(data)):
    start = end - WINDOW + 1
    xw = x[start:end+1]
    yw = y[start:end+1]

    alpha, beta, beta_se, t_stat, beta_p = ols_window(xw, yw)
    coint_stat, coint_p = safe_coint_p(yw, xw)

    rows.append({
        "date": dates[end],
        "alpha": alpha,
        "beta": beta,
        "beta_se": beta_se,
        "beta_t": t_stat,
        "beta_p": beta_p,
        "coint_stat": coint_stat,
        "coint_p": coint_p,
    })

roll = pd.DataFrame(rows).set_index("date")
roll.head()


In [ ]:
# Future rolling-window coefficient movement
roll["next_abs_beta_move"] = (roll["beta"].shift(-1) - roll["beta"]).abs()

future_moves = pd.concat(
    {
        h: (roll["beta"].shift(-h) - roll["beta"]).abs()
        for h in range(1, PRED_HORIZON + 1)
    },
    axis=1,
)
roll["future_avg_beta_move"] = future_moves.mean(axis=1)
roll["future_max_beta_move"] = future_moves.max(axis=1)

# Large-movement label is defined ex post only for evaluation.
q_move = roll["future_avg_beta_move"].quantile(LARGE_MOVE_Q)
roll["large_future_move"] = roll["future_avg_beta_move"] >= q_move

roll.tail()


## 1. Coefficient fluctuation

The confidence band is the usual pointwise OLS uncertainty band. It visualizes how periods with high coefficient uncertainty coincide with wider uncertainty around the rolling estimate; it is not a structural-stability confidence band.


In [ ]:
fig, ax = plt.subplots()
valid = roll[["beta", "beta_se"]].dropna()
line = ax.plot(valid.index, valid["beta"], lw=1.5, label=r"Rolling $\hat{\beta}_t$")[0]
ax.fill_between(
    valid.index,
    valid["beta"] - 1.96 * valid["beta_se"],
    valid["beta"] + 1.96 * valid["beta_se"],
    alpha=0.16,
    color=line.get_color(),
    label=r"$\hat{\beta}_t \pm 1.96 SE_t$",
)
ax.set_title(f"Rolling coefficient fluctuation ({Y_TICKER} on {X_TICKER}, window={WINDOW})")
ax.set_ylabel(r"$\hat{\beta}_t$")
ax.set_xlabel("")
ax.legend(frameon=False, ncol=2)
fig.tight_layout()
fig.savefig(OUT / "01_beta_fluctuation.png", dpi=220, bbox_inches="tight")
plt.show()


## 2. Cointegration p-value fluctuation

This is the rolling Engle-Ganger residual-based cointegration p-value. The 5% line is shown only as a familiar reference threshold; the main point of the figure is how strongly the test result can fluctuate as the window advances.


In [ ]:
fig, ax = plt.subplots()
valid = roll["coint_p"].dropna()
ax.plot(valid.index, valid, lw=1.25, label="Engle-Granger p-value")
ax.axhline(0.05, ls="--", lw=1.25, label="5% threshold")
ax.set_ylim(0, 1)
ax.set_title(f"Rolling cointegration p-value fluctuation (window={WINDOW})")
ax.set_ylabel("p-value")
ax.set_xlabel("")
ax.legend(frameon=False)
fig.tight_layout()
fig.savefig(OUT / "02_cointegration_pvalue_fluctuation.png", dpi=220, bbox_inches="tight")
plt.show()


## 3. Does current SE predict future beta movement?

The target is fixed in advance as

\[
S_t = 
rac{1}{H}\sum_{h=1}^{H}|\hateta_{t+h}-\hateta_t|.
\]

The first result plot groups observations by current SE decile. A useful predictor should show larger subsequent coefficient movement in higher-SE bins.


In [ ]:
eval_df = roll[["beta_se", "beta_p", "beta_t", "future_avg_beta_move", "large_future_move"]].dropna().copy()

eval_df["se_decile"] = pd.qcut(
    eval_df["beta_se"], q=SE_BINS, labels=False, duplicates="drop"
) + 1

bin_summary = eval_df.groupby("se_decile", observed=True).agg(
    mean_se=("beta_se", "mean"),
    mean_future_move=("future_avg_beta_move", "mean"),
    prob_large_move=("large_future_move", "mean"),
    n=("large_future_move", "size"),
).reset_index()

rho_spearman = eval_df["beta_se"].rank().corr(eval_df["future_avg_beta_move"].rank())
rho_pearson = eval_df["beta_se"].corr(eval_df["future_avg_beta_move"])

fig, ax = plt.subplots(figsize=(8.5, 5.0))
ax.plot(
    bin_summary["se_decile"],
    bin_summary["mean_future_move"],
    marker="o",
    lw=2,
)
ax.set_xticks(bin_summary["se_decile"])
ax.set_xlabel("Current SE decile (low → high)")
ax.set_ylabel(r"Mean future $|\hat{\beta}_{t+h}-\hat{\beta}_t|$")
ax.set_title(
    "Higher current SE predicts larger subsequent coefficient movement
"
    f"Spearman ρ={rho_spearman:.2f}; Pearson r={rho_pearson:.2f}"
)
fig.tight_layout()
fig.savefig(OUT / "03_se_decile_future_beta_movement.png", dpi=220, bbox_inches="tight")
plt.show()

bin_summary


In [ ]:
# Probability version: easy to use as a single results slide
fig, ax = plt.subplots(figsize=(8.5, 5.0))
ax.bar(bin_summary["se_decile"], 100 * bin_summary["prob_large_move"])
ax.axhline(
    100 * eval_df["large_future_move"].mean(),
    ls="--",
    lw=1.25,
    label="Unconditional top-20% rate",
)
ax.set_xticks(bin_summary["se_decile"])
ax.set_xlabel("Current SE decile (low → high)")
ax.set_ylabel("Probability of large future beta movement (%)")
ax.set_title("Large future coefficient movements are concentrated in high-SE periods")
ax.legend(frameon=False)
fig.tight_layout()
fig.savefig(OUT / "04_se_decile_large_move_probability.png", dpi=220, bbox_inches="tight")
plt.show()


In [ ]:
def auc_manual(y_true, score):
    y_true = np.asarray(y_true, dtype=int)
    score = np.asarray(score, dtype=float)
    mask = np.isfinite(score)
    y_true, score = y_true[mask], score[mask]
    if len(np.unique(y_true)) < 2:
        return np.nan
    ranks = pd.Series(score).rank(method="average").to_numpy()
    n_pos = (y_true == 1).sum()
    n_neg = (y_true == 0).sum()
    rank_sum_pos = ranks[y_true == 1].sum()
    return (rank_sum_pos - n_pos * (n_pos + 1) / 2) / (n_pos * n_neg)

auc = pd.Series({
    "SE(beta)": auc_manual(eval_df["large_future_move"], eval_df["beta_se"]),
    "beta p-value": auc_manual(eval_df["large_future_move"], eval_df["beta_p"]),
    "-|t|": auc_manual(eval_df["large_future_move"], -eval_df["beta_t"].abs()),
})

fig, ax = plt.subplots(figsize=(7.5, 4.8))
ax.bar(auc.index, auc.values)
ax.axhline(0.5, ls="--", lw=1.25, label="No discrimination")
ax.set_ylim(0.45, max(0.85, float(auc.max()) + 0.05))
ax.set_ylabel("AUC for top-20% future beta movement")
ax.set_title("Predictive ranking of coefficient-sensitivity measures")
ax.legend(frameon=False)
fig.tight_layout()
fig.savefig(OUT / "05_predictor_auc_comparison.png", dpi=220, bbox_inches="tight")
plt.show()

print(auc.sort_values(ascending=False))


In [ ]:
# Compact table for the results slide
q80_se = eval_df["beta_se"].quantile(0.80)
high_se = eval_df["beta_se"] >= q80_se

result_summary = pd.DataFrame({
    "quantity": [
        "Spearman corr(SE, future beta movement)",
        "Mean future movement — lower 80% SE",
        "Mean future movement — top 20% SE",
        "P(large future movement) — lower 80% SE",
        "P(large future movement) — top 20% SE",
        "AUC(SE -> large future movement)",
    ],
    "value": [
        rho_spearman,
        eval_df.loc[~high_se, "future_avg_beta_move"].mean(),
        eval_df.loc[high_se, "future_avg_beta_move"].mean(),
        eval_df.loc[~high_se, "large_future_move"].mean(),
        eval_df.loc[high_se, "large_future_move"].mean(),
        auc["SE(beta)"],
    ],
})

result_summary.to_csv(OUT / "result_summary.csv", index=False)
bin_summary.to_csv(OUT / "se_decile_summary.csv", index=False)
roll.to_csv(OUT / "rolling_results.csv")

result_summary
